# 04 Run Eval

Retrieval-focused pipeline and Query Analyzer evaluation. Build a draft CSV from canonical segment metadata, manually verify positives, then run metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -r /content/multimodal-search-demo/requirements-colab.txt

In [ ]:
import os, sys
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path('/content/multimodal-search-demo')
sys.path.insert(0, str(PROJECT_ROOT))
os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    print('GEMINI_API_KEY is not set. Analyzer will use rule-based fallback.')

SEGMENTS = PROJECT_ROOT / 'data' / 'processed' / 'canonical_segments.parquet'
EVAL_QUERIES = PROJECT_ROOT / 'data' / 'eval' / 'retrieval_eval_queries_draft.csv'
ADAPTER_PATH = '/content/drive/MyDrive/korean_cooking_shorts_dataset/siglip2_lora_qv_r16_best'

In [ ]:
import pandas as pd
from src.eval.build_retrieval_eval_template import build_query_specs_from_segments, build_retrieval_eval_template

segments = pd.read_parquet(SEGMENTS).fillna('')
specs = build_query_specs_from_segments(segments)
display(specs.groupby('query_type').size().reset_index(name='count'))
draft = build_retrieval_eval_template(segments, specs)
EVAL_QUERIES.parent.mkdir(parents=True, exist_ok=True)
draft.to_csv(EVAL_QUERIES, index=False, encoding='utf-8-sig')
display(draft[['query', 'query_type', 'auto_match_count', 'needs_review', 'notes']].head(20))

Review `data/eval/retrieval_eval_queries_draft.csv` before using the numbers in a report. The automatic positives are candidates, not final labels.

In [ ]:
import pandas as pd
from src.eval.run_eval import evaluate_queries_by_type, load_eval_queries
from src.eval.analyzer_eval import evaluate_analyzer
from src.index.qdrant_client import get_qdrant_client
from src.models.bge_encoder import BGEEncoder
from src.models.siglip_encoder import SigLIPEncoder

queries = load_eval_queries(EVAL_QUERIES)
detail, analyzer_summary = evaluate_analyzer(queries)
display(pd.DataFrame([analyzer_summary]))

client = get_qdrant_client()
bge = BGEEncoder()
siglip = SigLIPEncoder(adapter_path=ADAPTER_PATH)

rows = []
for mode in ['text-only', 'image-only', 'hybrid', 'unified']:
    rows.extend(evaluate_queries_by_type(queries, client, bge, siglip, mode))
pd.DataFrame(rows)